# How Do Canada's Big Six Banks React to Bank of Canada Rate Decisions?

**Question:** When the Bank of Canada changes its policy interest rate, how do the shares of Canada's six largest banks respond, and which bank is the most sensitive?

**Why it matters:** The Big Six (RBC, TD, Scotiabank, BMO, CIBC, National Bank) make up a large share of the S&P/TSX Composite and of most Canadian investors' portfolios. Interest rates drive their lending margins, loan losses and valuations, so knowing how each bank reacts to rate moves is useful for portfolio allocation and risk management.

### Approach
| Step | What | Tools |
|---|---|---|
| 1 | Policy rate history and every rate change | Bank of Canada **Valet API** (`requests`, JSON) |
| 2 | Context table of the Big Six | **Web scraping** of Wikipedia (`BeautifulSoup`, `pd.read_html`) |
| 3 | Daily share prices and dividends | **`yfinance`** |
| 4 | Combine the sources | `pandas` (`merge`, `merge_asof`) |
| 5 | Event study around each rate change + returns by rate cycle | `pandas`, `numpy` |
| 6 | Interactive dashboard | **Plotly** |

### Data sources
- Bank of Canada Valet API, series `V39079` (Target for the overnight rate)
- Yahoo Finance via `yfinance` (TSX tickers `RY.TO`, `TD.TO`, `BNS.TO`, `BMO.TO`, `CM.TO`, `NA.TO`, benchmark `^GSPTSE`)
- Wikipedia, *Big Six (banks)*

## 0. Setup

In [1]:
# Installs the libraries this notebook needs (works in VS Code, Jupyter and Colab)
%pip install -q pandas numpy requests yfinance beautifulsoup4 lxml plotly nbformat

import warnings
warnings.filterwarnings("ignore")

from io import StringIO

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from bs4 import BeautifulSoup
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option("display.float_format", "{:,.2f}".format)

START_DATE = "2000-01-01"
HEADERS = {"User-Agent": "canadian-banks-portfolio-project/1.0 (educational data analysis)"}

BANKS = {
    "RY.TO": "RBC",
    "TD.TO": "TD",
    "BNS.TO": "Scotiabank",
    "BMO.TO": "BMO",
    "CM.TO": "CIBC",
    "NA.TO": "National Bank",
}
BENCHMARK = "^GSPTSE"          # S&P/TSX Composite index
BANK_NAMES = list(BANKS.values())


def make_naive(index):
    # yfinance sometimes returns time-zone-aware dates; drop the time zone so all sources line up
    # and use one date precision (nanoseconds) everywhere so merges between sources work
    index = pd.to_datetime(index)
    if index.tz is not None:
        index = index.tz_localize(None)
    return index.astype("datetime64[ns]")


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 1. Policy rate from the Bank of Canada Valet API

The Valet API returns JSON. Each observation looks like `{"d": "2024-06-05", "V39079": {"v": "4.75"}}`: `d` is the date and `v` the value (as text).

In [2]:
def get_boc_series(series_code, start_date=START_DATE):
    # Download one time series from the Bank of Canada Valet API and return a tidy DataFrame
    url = f"https://www.bankofcanada.ca/valet/observations/{series_code}/json"
    response = requests.get(url, params={"start_date": start_date}, headers=HEADERS, timeout=30)
    response.raise_for_status()
    payload = response.json()

    rows = [
        {"date": obs["d"], "value": obs.get(series_code, {}).get("v")}
        for obs in payload["observations"]
    ]
    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"]).astype("datetime64[ns]")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna().sort_values("date").reset_index(drop=True)
    return df, payload.get("seriesDetail", {})


policy_rate, series_detail = get_boc_series("V39079")
policy_rate = policy_rate.rename(columns={"value": "policy_rate"})

print("Series:", series_detail.get("V39079", {}).get("label", "V39079"))
print(f"{len(policy_rate):,} observations from {policy_rate['date'].min():%Y-%m-%d} to {policy_rate['date'].max():%Y-%m-%d}")
policy_rate.tail()

Series: Target for the overnight rate (business daily)
4,565 observations from 2009-04-21 to 2026-11-06


,date,policy_rate
4560,2026-11-02,2.50
4561,2026-11-03,2.50
4562,2026-11-04,2.50
4563,2026-11-05,2.50
4564,2026-11-06,2.50


### Every rate change, and the rate "cycle" we are in

- A **rate decision** is any day the policy rate differs from the previous observation.
- The **cycle** on a given day is the direction of the most recent change: after a hike (and through any holds that follow) we are in a *Hiking cycle*; after a cut, a *Cutting cycle*.

In [3]:
policy_rate["change_bps"] = (policy_rate["policy_rate"].diff() * 100).round()

decisions = policy_rate[policy_rate["change_bps"].fillna(0) != 0].copy()
decisions["direction"] = np.where(decisions["change_bps"] > 0, "Hike", "Cut")
decisions = decisions[["date", "policy_rate", "change_bps", "direction"]].reset_index(drop=True)

last_move = np.sign(policy_rate["change_bps"].where(policy_rate["change_bps"] != 0)).ffill()
policy_rate["cycle"] = last_move.map({1.0: "Hiking cycle", -1.0: "Cutting cycle"}).fillna("Before first change")

print(f"{len(decisions)} rate changes: {(decisions['direction'] == 'Hike').sum()} hikes, "
      f"{(decisions['direction'] == 'Cut').sum()} cuts")
decisions.tail(10)

33 rate changes: 19 hikes, 14 cuts


,date,policy_rate,change_bps,direction
23,2024-06-06,4.75,-25.00,Cut
24,2024-07-25,4.50,-25.00,Cut
25,2024-09-05,4.25,-25.00,Cut
26,2024-10-24,3.75,-50.00,Cut
27,2024-12-12,3.25,-50.00,Cut
28,2025-01-30,3.00,-25.00,Cut
29,2025-03-13,2.75,-25.00,Cut
30,2025-09-18,2.50,-25.00,Cut
31,2025-10-30,2.25,-25.00,Cut
32,2026-10-29,2.50,25.00,Hike


## 2. Web scraping: the Big Six on Wikipedia

`BeautifulSoup` finds the `wikitable` tables on the page, and `pd.read_html` turns each one into a DataFrame. We keep the table that mentions Royal Bank. This is context for the reader; the analysis itself uses the tickers defined in the setup cell.

In [4]:
wiki_url = "https://en.wikipedia.org/wiki/Big_Six_(banks)"
wiki_banks = pd.DataFrame()

try:
    html = requests.get(wiki_url, headers=HEADERS, timeout=30).text
    soup = BeautifulSoup(html, "html.parser")
    print("Page title:", soup.title.text)

    wikitables = soup.find_all("table", class_="wikitable")
    tables = [pd.read_html(StringIO(str(table)))[0] for table in wikitables]
    print(f"Found {len(tables)} wikitable(s)")

    for table in tables:
        if table.astype(str).apply(lambda col: col.str.contains("Royal Bank", case=False)).any().any():
            wiki_banks = table
            break
except Exception as error:
    print("Could not scrape Wikipedia:", error)

wiki_banks.head(10)

Page title: Big Six (banks) - Wikipedia
Found 0 wikitable(s)


""


## 3. Share prices and dividends with yfinance

- **Adj Close** includes dividends and splits, so it is the right price for measuring **returns**.
- **Close** is the actual trading price, which is the right price for measuring **dividend yield**.

In [5]:
tickers = list(BANKS) + [BENCHMARK]
raw = yf.download(tickers, start=START_DATE, auto_adjust=False, progress=False)

rename_map = {**BANKS, BENCHMARK: "TSX"}
adj_close = raw["Adj Close"].rename(columns=rename_map)
close = raw["Close"].rename(columns=rename_map)
adj_close.index = make_naive(adj_close.index)
close.index = make_naive(close.index)

adj_close = adj_close[BANK_NAMES + ["TSX"]].dropna(how="all")
close = close[BANK_NAMES].dropna(how="all")

print(f"Prices from {adj_close.index.min():%Y-%m-%d} to {adj_close.index.max():%Y-%m-%d}, {len(adj_close):,} trading days")
adj_close.tail()

Prices from 2000-01-04 to 2026-11-09, 6,745 trading days


Ticker,RBC,TD,Scotiabank,BMO,CIBC,National Bank,TSX
Date,,,,,,,
2026-11-03,285.42,174.75,131.17,240.56,163.36,213.01,"35,532.30"
2026-11-04,292.57,178.93,134.50,247.61,167.24,218.96,"36,341.40"
2026-11-05,285.86,172.55,128.57,241.70,161.97,213.95,"35,605.30"
2026-11-06,284.16,172.92,129.54,241.42,162.69,214.61,"35,789.10"
2026-11-09,288.46,175.86,132.07,246.14,165.25,217.83,"36,476.37"


In [6]:
dividend_rows = []
for ticker, bank in BANKS.items():
    payments = yf.Ticker(ticker).dividends
    if payments.empty:
        continue
    payments.index = make_naive(payments.index)
    for year, amount in payments.groupby(payments.index.year).sum().items():
        dividend_rows.append({"bank": bank, "year": int(year), "dividend": amount})

dividends = pd.DataFrame(dividend_rows)
dividends = dividends[dividends["year"] < pd.Timestamp.today().year]   # drop the incomplete current year

# Average trading price per bank per year, reshaped to long format so it can be merged
avg_price = close.groupby(close.index.year).mean()
avg_price.index.name = "year"
avg_price = avg_price.reset_index().melt(id_vars="year", var_name="bank", value_name="avg_price")

dividends = dividends.merge(avg_price, on=["bank", "year"], how="left")
dividends["dividend_yield_pct"] = dividends["dividend"] / dividends["avg_price"] * 100
dividends.tail(12)

,bank,year,dividend,avg_price,dividend_yield_pct
174,National Bank,2014,1.92,47.58,4.04
175,National Bank,2015,2.08,45.48,4.57
176,National Bank,2016,2.20,44.65,4.93
177,National Bank,2017,2.32,57.44,4.04
178,National Bank,2018,2.49,62.26,4.00
179,National Bank,2019,2.72,64.10,4.24
180,National Bank,2020,2.84,64.81,4.38
181,National Bank,2021,3.00,91.58,3.28
182,National Bank,2022,3.68,93.48,3.94
183,National Bank,2023,4.07,96.58,4.21


## 4. Event study: how each bank moves around a rate change

For each rate change, day 0 is the first trading day on or after the change. We measure:

- **Event return:** close of day -1 to close of day +1
- **Week return:** close of day -1 to close of day +5
- **Excess return:** the bank's return minus the TSX return over the same window, which removes moves that hit the whole market

Each bank's results are combined with the TSX results using `merge` on the event date.

In [7]:
def window_return(prices, event_date, before=1, after=1):
    # Return from `before` trading days before day 0 to `after` trading days after it
    position = prices.index.searchsorted(event_date)
    start, end = position - before, position + after
    if start < 0 or end >= len(prices):
        return np.nan
    return prices.iloc[end] / prices.iloc[start] - 1


event_rows = []
for _, decision in decisions.iterrows():
    for name in BANK_NAMES + ["TSX"]:
        series = adj_close[name].dropna()
        event_rows.append({
            "date": decision["date"],
            "direction": decision["direction"],
            "change_bps": decision["change_bps"],
            "bank": name,
            "event_return": window_return(series, decision["date"], 1, 1),
            "week_return": window_return(series, decision["date"], 1, 5),
        })

all_events = pd.DataFrame(event_rows).dropna(subset=["event_return", "week_return"])

tsx_events = (all_events[all_events["bank"] == "TSX"][["date", "event_return", "week_return"]]
              .rename(columns={"event_return": "tsx_event_return", "week_return": "tsx_week_return"}))

events = all_events[all_events["bank"] != "TSX"].merge(tsx_events, on="date", how="inner")
events["excess_event"] = events["event_return"] - events["tsx_event_return"]
events["excess_week"] = events["week_return"] - events["tsx_week_return"]

event_summary = (events.groupby(["bank", "direction"])[["event_return", "excess_event", "week_return", "excess_week"]]
                 .mean().mul(100).round(2))
event_counts = events.groupby("direction")["date"].nunique()
print("Rate changes with price data:", event_counts.to_dict())
event_summary

Rate changes with price data: {'Cut': 14, 'Hike': 19}


event_return  excess_event  week_return  excess_week
bank          direction                                                      
BMO           Cut               -1.10         -0.71        -3.42        -0.74
              Hike               0.12          0.15         0.54         0.58
CIBC          Cut               -0.44         -0.06        -2.95        -0.28
              Hike               0.13          0.16         0.69         0.73
National Bank Cut               -1.34         -0.95        -4.42        -1.75
              Hike               0.26          0.29         1.00         1.05
RBC           Cut               -0.12          0.27        -3.10        -0.43
              Hike              -0.18         -0.15         0.33         0.37
Scotiabank    Cut               -0.74         -0.35        -3.09        -0.42
              Hike               0.33          0.36         0.32         0.36
TD            Cut                0.10          0.48        -2.56         0.11
              Hike               0.13          0.16         0.44         0.49

## 5. Returns by rate cycle

`merge_asof` attaches to each trading day the most recent policy-rate observation (and its cycle), even when the two calendars do not match exactly. We then compare annualized returns and volatility in hiking versus cutting cycles.

In [8]:
daily_returns = adj_close.pct_change().dropna(how="all")
daily_returns.index.name = "date"
daily_returns = daily_returns.reset_index()

daily = pd.merge_asof(daily_returns.sort_values("date"),
                      policy_rate[["date", "policy_rate", "cycle"]].sort_values("date"),
                      on="date", direction="backward")
daily = daily[daily["cycle"].isin(["Hiking cycle", "Cutting cycle"])]

long_returns = daily.melt(id_vars=["date", "cycle"], value_vars=BANK_NAMES + ["TSX"],
                          var_name="bank", value_name="daily_return").dropna()

cycle_summary = (long_returns.groupby(["bank", "cycle"])["daily_return"]
                 .agg(annual_return=lambda r: r.mean() * 252 * 100,
                      annual_volatility=lambda r: r.std() * np.sqrt(252) * 100,
                      trading_days="count")
                 .round(2))
cycle_summary

annual_return  annual_volatility  trading_days
bank          cycle                                                        
BMO           Cutting cycle          28.18              23.51          1723
              Hiking cycle            4.64              14.63          2403
CIBC          Cutting cycle          29.03              21.80          1723
              Hiking cycle            6.29              15.25          2403
National Bank Cutting cycle          24.19              24.79          1723
              Hiking cycle           13.98              14.45          2403
RBC           Cutting cycle          23.49              19.81          1723
              Hiking cycle            9.61              14.70          2403
Scotiabank    Cutting cycle          25.54              21.56          1723
              Hiking cycle            3.19              14.86          2403
TD            Cutting cycle          28.30              21.49          1723
              Hiking cycle            5.89              14.83          2403
TSX           Cutting cycle          13.30              17.10          1723
              Hiking cycle            4.14              12.19          2403

## 6. Dashboard

In [9]:
COLORS = px.colors.qualitative.Safe
bank_colors = {bank: COLORS[i % len(COLORS)] for i, bank in enumerate(BANK_NAMES)}
bank_colors["TSX"] = "#555555"

# Contiguous hiking periods, used as shaded bands
cycle_runs = policy_rate[["date", "cycle"]].copy()
cycle_runs["run"] = (cycle_runs["cycle"] != cycle_runs["cycle"].shift()).cumsum()
hiking_spans = (cycle_runs[cycle_runs["cycle"] == "Hiking cycle"]
                .groupby("run")["date"].agg(["min", "max"]).values)

# Figure 1: policy rate and growth of $100 in each bank
growth = adj_close / adj_close.bfill().iloc[0] * 100

fig_overview = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                             row_heights=[0.35, 0.65],
                             subplot_titles=("Bank of Canada policy rate (%)",
                                             "Growth of $100 invested (dividends reinvested)"))
fig_overview.add_trace(go.Scatter(x=policy_rate["date"], y=policy_rate["policy_rate"], line_shape="hv",
                                  name="Policy rate", line=dict(color="#222222")), row=1, col=1)
for direction, symbol, color in [("Hike", "triangle-up", "#c0392b"), ("Cut", "triangle-down", "#2471a3")]:
    subset = decisions[decisions["direction"] == direction]
    fig_overview.add_trace(go.Scatter(x=subset["date"], y=subset["policy_rate"], mode="markers", name=direction,
                                      marker=dict(symbol=symbol, size=9, color=color)), row=1, col=1)
for name in BANK_NAMES + ["TSX"]:
    fig_overview.add_trace(go.Scatter(x=growth.index, y=growth[name], name=name,
                                      line=dict(color=bank_colors[name], width=2.5 if name == "TSX" else 1.5,
                                                dash="dot" if name == "TSX" else "solid")), row=2, col=1)
for start, end in hiking_spans:
    fig_overview.add_vrect(x0=start, x1=end, fillcolor="#c0392b", opacity=0.08, line_width=0, row="all", col=1)
fig_overview.update_layout(height=720, title="Policy rate and Big Six performance (red bands = hiking cycles)",
                           hovermode="x unified", template="plotly_white")
fig_overview.show()

In [10]:
# Figure 2: average excess return in the week after a hike vs a cut
week_plot = event_summary.reset_index()
fig_events = px.bar(week_plot, x="bank", y="excess_week", color="direction", barmode="group",
                    color_discrete_map={"Hike": "#c0392b", "Cut": "#2471a3"},
                    labels={"excess_week": "Avg. excess return vs TSX, day -1 to +5 (%)", "bank": ""},
                    title="Which bank beats the market after a rate move?")
fig_events.add_hline(y=0, line_color="#888888")
fig_events.update_layout(template="plotly_white", height=450)
fig_events.show()

# Figure 3: annualized return in hiking vs cutting cycles
cycle_plot = cycle_summary.reset_index()
fig_cycles = px.bar(cycle_plot, x="bank", y="annual_return", color="cycle", barmode="group",
                    color_discrete_map={"Hiking cycle": "#c0392b", "Cutting cycle": "#2471a3"},
                    labels={"annual_return": "Annualized return (%)", "bank": ""},
                    title="Annualized return during hiking vs cutting cycles")
fig_cycles.add_hline(y=0, line_color="#888888")
fig_cycles.update_layout(template="plotly_white", height=450)
fig_cycles.show()

# Figure 4: dividend yield over time
fig_dividends = px.line(dividends, x="year", y="dividend_yield_pct", color="bank", markers=True,
                        color_discrete_map=bank_colors,
                        labels={"dividend_yield_pct": "Dividend yield (%)", "year": ""},
                        title="Annual dividend yield of the Big Six")
fig_dividends.update_layout(template="plotly_white", height=450)
fig_dividends.show()

## 7. Key findings (generated from the data)

In [11]:
first_date, last_date = adj_close.index.min(), adj_close.index.max()
week = event_summary["excess_week"].unstack("direction")

lines = [
    f"Period analysed: {first_date:%B %Y} to {last_date:%B %Y}, covering {event_counts.sum()} Bank of Canada rate changes "
    f"({event_counts.get('Hike', 0)} hikes, {event_counts.get('Cut', 0)} cuts).",
]
if "Hike" in week:
    lines.append(f"After hikes, {week['Hike'].idxmax()} did best against the TSX over the following week "
                 f"({week['Hike'].max():+.2f}% on average) and {week['Hike'].idxmin()} did worst ({week['Hike'].min():+.2f}%).")
if "Cut" in week:
    lines.append(f"After cuts, {week['Cut'].idxmax()} held up best against the TSX ({week['Cut'].max():+.2f}%) "
                 f"and {week['Cut'].idxmin()} did worst ({week['Cut'].min():+.2f}%).")
if {"Hike", "Cut"}.issubset(week.columns):
    spread = (week["Hike"] - week["Cut"]).abs()
    lines.append(f"Most rate-sensitive bank (largest gap between its reaction to hikes and to cuts): "
                 f"{spread.idxmax()} ({spread.max():.2f} percentage points).")

cyc = cycle_summary["annual_return"].unstack("cycle")
if {"Hiking cycle", "Cutting cycle"}.issubset(cyc.columns):
    banks_only = cyc.drop(index="TSX", errors="ignore")
    lines.append(f"Across whole cycles, {banks_only['Hiking cycle'].idxmax()} had the highest annualized return in hiking cycles "
                 f"({banks_only['Hiking cycle'].max():.1f}%) and {banks_only['Cutting cycle'].idxmax()} in cutting cycles "
                 f"({banks_only['Cutting cycle'].max():.1f}%). The TSX returned {cyc.loc['TSX', 'Hiking cycle']:.1f}% and "
                 f"{cyc.loc['TSX', 'Cutting cycle']:.1f}% respectively.")

recent = dividends[dividends["year"] >= dividends["year"].max() - 4]
if not recent.empty:
    top_yield = recent.groupby("bank")["dividend_yield_pct"].mean()
    lines.append(f"Highest average dividend yield over the last five full years: {top_yield.idxmax()} ({top_yield.max():.2f}%).")

for i, line in enumerate(lines, 1):
    print(f"{i}. {line}")

1. Period analysed: January 2000 to November 2026, covering 33 Bank of Canada rate changes (19 hikes, 14 cuts).
2. After hikes, National Bank did best against the TSX over the following week (+1.05% on average) and Scotiabank did worst (+0.36%).
3. After cuts, TD held up best against the TSX (+0.11%) and National Bank did worst (-1.75%).
4. Most rate-sensitive bank (largest gap between its reaction to hikes and to cuts): National Bank (2.80 percentage points).
5. Across whole cycles, National Bank had the highest annualized return in hiking cycles (14.0%) and CIBC in cutting cycles (29.0%). The TSX returned 4.1% and 13.3% respectively.
6. Highest average dividend yield over the last five full years: Scotiabank (5.56%).


### Limitations
- **Correlation, not causation.** Other news (earnings, the global market) also moves bank shares around decision dates. Excess returns versus the TSX reduce, but do not remove, this effect.
- **Expected moves.** Markets usually price in rate decisions before they happen, so part of the reaction can occur before day -1.
- **Cycle definition.** A cycle lasts from one move until the next move in the other direction, including long holds.
- **Dates.** Day 0 is the first trading day on or after the date the policy-rate series changes.

## 8. Save the results

In [12]:
import os
os.makedirs("data", exist_ok=True)

policy_rate.to_csv("data/policy_rate.csv", index=False)
decisions.to_csv("data/rate_decisions.csv", index=False)
events.to_csv("data/event_returns.csv", index=False)
event_summary.to_csv("data/event_summary.csv")
cycle_summary.to_csv("data/cycle_summary.csv")
dividends.to_csv("data/dividends.csv", index=False)

# One self-contained HTML dashboard for the portfolio
with open("dashboard.html", "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><title>Big Six banks and Bank of Canada rates</title></head><body>")
    f.write("<h1 style='font-family:sans-serif'>How Canada's Big Six banks react to Bank of Canada rate decisions</h1>")
    for i, fig in enumerate([fig_overview, fig_events, fig_cycles, fig_dividends]):
        f.write(fig.to_html(full_html=False, include_plotlyjs="cdn" if i == 0 else False))
    f.write("</body></html>")

print("Saved CSV files in data/ and dashboard.html")

Saved CSV files in data/ and dashboard.html
